# 05 — Reactive Observers

Agents are passive until an observer makes them reactive. lionag2 uses AG2's `@agent.observer(EventType)` to wire reactive coordination:

- When a `FindingEmitted` with high novelty arrives → observer spawns a child team at depth+1.
- When a `PaperGapEvent` fires → observer triggers depth expansion to fill the gap.
- When a `ContradictionFound` arrives → observer records it for the cross-check.

Observers can be **sync or async**. They fire every time the agent emits a matching event type.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent, MemoryStream
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import BaseEvent, ToolResultsEvent
from autogen.beta.tools import ExaToolkit

config = OpenAIConfig("gpt-5.4-mini", api_key=os.getenv("OPENAI_API_KEY"))

## Observer — react to tool results

`@agent.observer(ToolResultsEvent)` fires every time the agent gets tool results back. This is how lionag2 captures URLs from Exa search — the same pattern from notebook 01, but here we'll see it drive a real research workflow.

In [2]:
exa = ExaToolkit(api_key=os.getenv("EXA_API_KEY"))
captured_urls: dict[str, str] = {}

agent = Agent(
    "surveyor",
    prompt="Search for papers and summarize findings.",
    config=config,
    tools=[exa],
)


@agent.observer(ToolResultsEvent)
def capture_urls(event: ToolResultsEvent) -> None:
    for r in event.results:
        result = getattr(r, "result", None)
        if not result:
            continue
        for part in getattr(result, "parts", []):
            data = getattr(part, "data", None)
            if not data:
                continue
            for hit in getattr(data, "results", None) or []:
                t, u = getattr(hit, "title", None), getattr(hit, "url", None)
                if t and u:
                    captured_urls[t] = u


stream = MemoryStream()
reply = await agent.ask(
    "Find 3 papers on reactive multi-agent coordination. Give title and one key finding.",
    stream=stream,
)

print(f"Agent reply: {reply.body[:300]}")
print(f"\nURLs captured by observer: {len(captured_urls)}")
for title, url in list(captured_urls.items())[:5]:
    print(f"  [{title[:60]}]({url})")

## The reactive chain

In the engine, observers on each agent check novelty against a threshold. High-novelty findings trigger depth expansion — spawning a child team at depth+1.

```python
@agent.observer(FindingEmitted)
def _on_finding(event: FindingEmitted) -> None:
    if event.novelty >= self.novelty_threshold:
        self._spawn(self._spawn_depth_node(
            event.claim, parent_node_id=node_id, depth=depth + 1
        ))
```

The research tree **emerges from observer reactions**, not from an imperative BFS loop.

In [3]:
# Count events in the stream to see what happened during the turn
events = await stream.history.get_events()
from collections import Counter

counts = Counter(type(e).__name__ for e in events)
print(f"Events in stream: {len(events)}")
for name, count in counts.most_common():
    print(f"  {name}: {count}")

print(f"\nThe observer fired {len(captured_urls)} times during the agent's turn.")
print("In lionag2's engine, this same pattern drives depth expansion:")
print("  FindingEmitted(novelty=0.9) → observer → spawn child team")

## Depth expansion control

The engine controls depth expansion with:

- **`novelty_threshold`** (default 0.7) — only findings above this trigger children.
- **`max_depth`** — caps the recursion tree.
- **`max_concurrent`** — limits concurrent depth nodes.
- **Topic deduplication** — `_seen_topics` prevents re-investigating the same question.

## All observers in the engine

Each agent created by `_make_agent()` gets these observers:

```python
@agent.observer(FindingEmitted)
def _on_finding(event):      # novelty → depth expansion

@agent.observer(DepthRequested)
def _on_depth(event):        # explicit depth request → spawn child

@agent.observer(ContradictionFound)
def _on_contradiction(event): # record for cross-check

@agent.observer(PivotDetected)
def _on_pivot(event):        # record for cross-check

@agent.observer(HandoffRequested)
def _on_handoff(event):      # agent-to-agent routing

@agent.observer(ToolResultsEvent)
def _on_tools(event):        # URL capture from Exa
```

The paper writer gets an additional `PaperGapEvent` observer for gap→depth feedback.

## Up next

Agents need bounded context to stay effective over long conversations. Tutorial 06 introduces assembly policies and the knowledge store.